# Data Pipeline: Split, Leakage Prevention, Dataset Classes

**Standalone notebook** - clones the repo fresh and re-downloads the primary
dataset, so it doesn't depend on any other notebook having been run in this
session.

**What this notebook does:**
1. Builds the canonical 569-image file list (excluding `Positive/Labelled`)
2. Detects exact/near-duplicate images, to use as a leakage-prevention
   grouping key (no patient/slide ID exists in this dataset - see
   `docs/dataset_findings.md`)
3. Carves out the held-out test set (touched once, at the very end)
4. Builds grouped 5-fold CV assignments on the development set (for model
   selection later)
5. Builds a small final train/val split on the full development set (for
   early stopping when training the chosen configuration - not a second
   selection step)
6. Saves everything to `results/splits/primary_dataset_splits.csv`
7. Demonstrates the `SickleCellDataset` class and augmentation pipeline with
   a quick visual sanity check

**Every design decision here (leakage strategy, validation/CV relationship,
class-imbalance approach, why each augmentation is biologically reasonable)
is written up in `docs/data_pipeline.md` - this notebook runs the code, that
file explains the reasoning.**

**No GPU needed for this notebook** - it's data prep only, no model
training happens here.


## 1. Get the project code and dependencies

In [ ]:
import os

REPO_URL = "https://github.com/aural0i/Sickle-cell-detection"
BRANCH = "claude/sickle-cell-cnn-research-g6fipc"
if not os.path.isdir("/content/Sickle-cell-detection"):
    !git clone --branch {BRANCH} {REPO_URL} /content/Sickle-cell-detection
else:
    !git -C /content/Sickle-cell-detection pull
%cd /content/Sickle-cell-detection


In [ ]:
# Same approach as notebook 01: skip reinstalling torch/torchvision (Colab's
# preinstalled GPU-matched pair), install everything else from requirements.txt
with open("requirements.txt") as f:
    lines = [l for l in f if not l.strip().startswith(("torch==", "torch>=", "torchvision=="))]
with open("/tmp/requirements_colab.txt", "w") as f:
    f.writelines(lines)

!pip install -q -r /tmp/requirements_colab.txt
print("Dependencies installed.")


## 2. Download the primary dataset

Same as notebook 01 - paste your Kaggle API token (`KGAT_...`) when
prompted, via https://www.kaggle.com/settings/api.


In [ ]:
from getpass import getpass

os.environ["KAGGLE_API_TOKEN"] = getpass("Paste your Kaggle API token (KGAT_...): ")

import kaggle
kaggle.api.authenticate()
print("Kaggle credentials accepted.")


In [ ]:
os.makedirs("data/train_source", exist_ok=True)
kaggle.api.dataset_download_files(
    "florencetushabe/sickle-cell-disease-dataset",
    path="data/train_source",
    unzip=True,
)
print("Download complete.")


## 3. Build the canonical file list

Excludes `Positive/Labelled` (annotation artifacts baked into the pixels -
see `docs/dataset_findings.md`). Uses only `Positive/Unlabelled` (422,
label=1/positive) and `Negative/Clear` (147, label=0/negative).


In [ ]:
import sys
sys.path.insert(0, "/content/Sickle-cell-detection")

from src.data import build_primary_manifest

manifest = build_primary_manifest("data/train_source")
print(f"Total images: {len(manifest)}")
print(manifest["label_name"].value_counts())
manifest.head()


## 4. Detect exact/near-duplicate images

This is our substitute for patient/slide grouping (see
`docs/data_pipeline.md`). Takes under a minute for 569 images.


In [ ]:
from src.data import find_duplicate_groups

group_ids, exact_hashes, phashes = find_duplicate_groups(manifest["path"])
manifest["group_id"] = manifest["path"].map(group_ids)

n_images = len(manifest)
n_groups = manifest["group_id"].nunique()
print(f"{n_images} images -> {n_groups} distinct groups")
if n_groups < n_images:
    dup_groups = manifest.groupby("group_id").filter(lambda g: len(g) > 1)
    print(f"Found {n_images - n_groups} images sharing a group with at least one other image:")
    print(dup_groups.sort_values("group_id")[["path", "label_name", "group_id"]])
else:
    print("No exact or near duplicates found - every image is its own group.")


## 5. Held-out test split (touch this exactly once, at final evaluation)

In [ ]:
from src.data import make_held_out_test_split

dev_df, test_df = make_held_out_test_split(manifest, n_splits=5, seed=42)
print(f"Development set: {len(dev_df)} images")
print(dev_df["label_name"].value_counts())
print()
print(f"Held-out test set: {len(test_df)} images")
print(test_df["label_name"].value_counts())

assert set(dev_df["group_id"]) & set(test_df["group_id"]) == set(), "group leakage between dev and test!"
print()
print("Confirmed: no duplicate-group overlap between development and held-out test.")


## 6. Grouped 5-fold CV on the development set (for model-selection later)

In [ ]:
from src.data import make_cv_folds

dev_df = make_cv_folds(dev_df, n_splits=5, seed=43)
print("CV fold sizes:")
print(dev_df["cv_fold"].value_counts().sort_index())
print()
print("Class balance per fold:")
print(dev_df.groupby("cv_fold")["label_name"].value_counts().unstack())


## 7. Final train/val split on the full development set (early stopping only)

Not a second model-selection step - see `docs/data_pipeline.md`.


In [ ]:
from src.data import make_final_train_val_split

dev_df = make_final_train_val_split(dev_df, val_frac=0.15, seed=44)
print("Final train/val sizes:")
print(dev_df["final_split"].value_counts())
print()
print("Class balance:")
print(dev_df.groupby("final_split")["label_name"].value_counts().unstack())


## 8. Save the split assignment for reproducibility

In [ ]:
import pandas as pd

test_df = test_df.copy()
test_df["cv_fold"] = -1
test_df["final_split"] = "held_out_test"

full = pd.concat([dev_df, test_df], ignore_index=True)

os.makedirs("results/splits", exist_ok=True)
out_path = "results/splits/primary_dataset_splits.csv"
full.to_csv(out_path, index=False)
print(f"Saved {len(full)} rows to {out_path}")
full.sample(5, random_state=0)


## 9. Class imbalance: compute weights (from the final-training training split only)

In [ ]:
from src.data import compute_class_weights

train_labels = dev_df.loc[dev_df["final_split"] == "train", "label"]
weights = compute_class_weights(train_labels)
print("Class weights [negative, positive]:", weights.tolist())
print("(higher weight = rarer class counts more toward the loss)")


## 10. Dataset class + augmentation sanity check

Shows a few augmented training examples side by side with un-augmented
validation examples, so we can visually confirm the augmentations look
reasonable before using them for real training.


In [ ]:
import matplotlib.pyplot as plt
from src.data import SickleCellDataset

IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD = [0.229, 0.224, 0.225]

def denormalize(tensor):
    img = tensor.clone()
    for c in range(3):
        img[c] = img[c] * IMAGENET_STD[c] + IMAGENET_MEAN[c]
    return img.clamp(0, 1).permute(1, 2, 0).numpy()

train_split = dev_df[dev_df["final_split"] == "train"].reset_index(drop=True)
val_split = dev_df[dev_df["final_split"] == "val"].reset_index(drop=True)

train_ds = SickleCellDataset(train_split, train=True)
val_ds = SickleCellDataset(val_split, train=False)

fig, axes = plt.subplots(2, 5, figsize=(15, 6))
for i in range(5):
    img, label = train_ds[i]
    axes[0, i].imshow(denormalize(img))
    axes[0, i].set_title(f"train (aug), label={label.item()}")
    axes[0, i].axis("off")

    img, label = val_ds[i]
    axes[1, i].imshow(denormalize(img))
    axes[1, i].set_title(f"val (no aug), label={label.item()}")
    axes[1, i].axis("off")

plt.tight_layout()
plt.savefig("results/splits/augmentation_sanity_check.png", dpi=100)
plt.show()
print("Saved a copy to results/splits/augmentation_sanity_check.png")


## 11. What to do with this output

Copy back to Claude:
- The duplicate-detection results (Section 4) - especially if any
  duplicates were found
- The split sizes and class balance from Sections 5-7
- Whether the augmented examples in Section 10 look reasonable (correctly
  rotated/flipped/brightness-adjusted, no weird artifacts)
- If you'd like `results/splits/primary_dataset_splits.csv` committed to
  the repo as a convenience artifact, send that file back too (it's tiny -
  just paths and labels, not images)

This is Step 5 (data pipeline) from the original plan. Next is a checkpoint
before moving into full model training (Step 6) - nothing trains yet.
